In [ ]:
# ============ 셀 1: 설정 (L5B: L5A 모델 6개 → Muon 파인튜닝) ============
import os, json, time, math, random, glob, shutil
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F

assert torch.cuda.is_available(), "GPU가 꺼져 있어요. Settings → Accelerator → GPU T4 로 바꾸세요."
DEV = 'cuda'

OUT  = '/kaggle/working/exp_l5_finetune'
DATA = '/tmp/llm_data'
os.makedirs(OUT, exist_ok=True); os.makedirs(DATA, exist_ok=True)

CFG = dict(
    seed=0, ctx=256, vocab=50304,
    d_model=512, n_layer=6, n_head=8, grad_clip=1.0,
    muon_momentum=0.95, ns_steps=5,
    muon_pre_lr=0.005,                 # L5A 사전학습 Muon lr
    # 파인튜닝: Muon(블록 안 행렬) + 보조 AdamW(임베딩 등), SAM 없음 — L3와 같은 규칙
    ft_muon_div=[10, 3],               # Muon lr = 사전학습 lr ÷ 10 (약), ÷ 3 (강)
    ft_adam_lrs=[1e-4, 1e-3 / 3],      # 보조 AdamW lr = L1·L2·L3와 동일
    ft_steps=1000, ft_bs=32, ft_wd=0.1, ft_warmup=50, ft_seeds=[0, 1, 2], ft_eval_every=100,
    code_files=10000, code_max_chars=20000,
    eval_seqs=400, eval_bs=32,
)
json.dump(CFG, open(f'{OUT}/config.json', 'w'), indent=2)

MODELS = ['pre_off', 'pre_decay_euc', 'pre_decay_spec_0.0005', 'pre_decay_spec_0.002',
          'pre_decay_spec_0.008', 'pre_full_spec']
SHORT = {'pre_off': 'off', 'pre_decay_euc': 'decay euc', 'pre_decay_spec_0.0005': 'decay spec 5e-4',
         'pre_decay_spec_0.002': 'decay spec 2e-3', 'pre_decay_spec_0.008': 'decay spec 8e-3',
         'pre_full_spec': 'full spec 8e-3'}

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

print('GPU :', torch.cuda.get_device_name(0), '| torch', torch.__version__)

In [ ]:
# ============ 셀 2: BabyLM + 파이썬 코드 다운로드, 토큰화 (L1~L3와 같은 방식) ============
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('gpt2'); tok.model_max_length = int(1e9)
EOS = tok.eos_token_id

def encode_texts(texts, bs=256):
    parts = []
    for i in range(0, len(texts), bs):
        for ids in tok(texts[i:i + bs])['input_ids']:
            parts.append(np.array(ids + [EOS], dtype=np.uint16))
    return np.concatenate(parts)

def chunk_lines(text, n=200):
    lines = text.split('\n')
    return ['\n'.join(lines[i:i + n]) for i in range(0, len(lines), n)]

# ---------- BabyLM ----------
if os.path.exists(f'{DATA}/babylm_val.npy'):
    print('BabyLM 토큰 파일이 이미 있음 → 건너뜀')
else:
    try:
        from huggingface_hub import snapshot_download
        root = snapshot_download('cambridge-climb/BabyLM', repo_type='dataset', allow_patterns=['10M/*', 'dev/*'])
        tr_files = sorted(f for f in glob.glob(f'{root}/10M/*') if os.path.isfile(f))
        dv_files = sorted(f for f in glob.glob(f'{root}/dev/*') if os.path.isfile(f))
        assert tr_files and dv_files, '파일 없음'
        tr_texts, dv_texts = [], []
        for f in tr_files: tr_texts += chunk_lines(open(f, encoding='utf-8', errors='ignore').read())
        for f in dv_files: dv_texts += chunk_lines(open(f, encoding='utf-8', errors='ignore').read()[:500_000])
        src = 'cambridge-climb/BabyLM (2023)'
    except Exception as e:
        print('2023판 다운로드 실패 → 2026판으로 대체:', repr(e)[:200])
        from datasets import load_dataset
        ds = load_dataset('BabyLM-community/BabyLM-2026-Strict-Small', split='train')
        col = [c for c, t in ds.features.items() if getattr(t, 'dtype', None) == 'string'][0]
        lines = [t for t in ds[col] if t and t.strip()]
        idx = np.random.RandomState(0).permutation(len(lines)); nv = len(lines) // 20
        dv_lines = [lines[i] for i in idx[:nv]]; tr_lines = [lines[i] for i in idx[nv:]]
        tr_texts = ['\n'.join(tr_lines[i:i + 200]) for i in range(0, len(tr_lines), 200)]
        dv_texts = ['\n'.join(dv_lines[i:i + 200]) for i in range(0, len(dv_lines), 200)]
        src = 'BabyLM-community/BabyLM-2026-Strict-Small (5%를 검증용으로 분리)'
    print('BabyLM 출처:', src, flush=True)
    tr = encode_texts(tr_texts); dv = encode_texts(dv_texts)
    np.save(f'{DATA}/babylm_train.npy', tr); np.save(f'{DATA}/babylm_val.npy', dv)
    print(f'BabyLM 토큰: train {len(tr)/1e6:.1f}M, val {len(dv)/1e6:.1f}M   (기존: 15.7M / 0.8M)')

# ---------- 파이썬 코드 ----------
if os.path.exists(f'{DATA}/code_val.npy'):
    print('코드 토큰 파일이 이미 있음 → 건너뜀')
else:
    from datasets import load_dataset
    try:
        ds = load_dataset('bigcode/the-stack-smol', data_dir='data/python', split='train')
        src = 'bigcode/the-stack-smol (python)'
    except Exception as e:
        print('the-stack-smol 실패 → codeparrot으로 대체:', repr(e)[:200])
        ds = load_dataset('codeparrot/codeparrot-clean-valid', split='train')
        src = 'codeparrot/codeparrot-clean-valid'
    n = min(CFG['code_files'], len(ds))
    ds = ds.shuffle(seed=0).select(range(n))
    files = [c[:CFG['code_max_chars']] for c in ds['content']]
    nv = n // 10
    cv = encode_texts(files[:nv]); ct = encode_texts(files[nv:])
    np.save(f'{DATA}/code_train.npy', ct); np.save(f'{DATA}/code_val.npy', cv)
    print(f'코드 출처: {src} | 토큰: train {len(ct)/1e6:.1f}M, val {len(cv)/1e6:.1f}M   (기존: 26.9M / 3.0M)')

In [ ]:
# ============ 셀 3: 모델, Muon, 파인튜닝 도구 ============
def load_tokens(name):
    return torch.from_numpy(np.load(f'{DATA}/{name}.npy').astype(np.int64)).to(DEV)

TR_CODE, VA_CODE, VA_BABY = load_tokens('code_train'), load_tokens('code_val'), load_tokens('babylm_val')

class Stream:
    def __init__(self, tokens, bs, ctx=CFG['ctx']):
        n = (len(tokens) - 1) // ctx
        self.chunks = tokens[:n * ctx + 1].unfold(0, ctx + 1, ctx)
        self.n, self.bs, self.spe = n, bs, n // bs
        self._key = None
    def batch(self, step, seed):
        ep, i = divmod(step, self.spe)
        if self._key != (seed, ep):
            g = torch.Generator().manual_seed(seed * 100003 + ep)
            self._perm = torch.randperm(self.n, generator=g).to(DEV); self._key = (seed, ep)
        b = self.chunks[self._perm[i * self.bs:(i + 1) * self.bs]]
        return b[:, :-1], b[:, 1:]

def make_eval(tokens, n, seed, ctx=CFG['ctx']):
    starts = np.random.RandomState(seed).randint(0, len(tokens) - ctx - 1, size=n)
    idx = torch.from_numpy(starts).to(DEV)[:, None] + torch.arange(ctx + 1, device=DEV)
    b = tokens[idx]
    return b[:, :-1], b[:, 1:]

EV_BABY = make_eval(VA_BABY, CFG['eval_seqs'], seed=11)    # 이전 실험들과 같은 평가 문장
EV_CODE = make_eval(VA_CODE, CFG['eval_seqs'], seed=12)

class Block(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.h = h
        self.ln1 = nn.LayerNorm(d); self.qkv = nn.Linear(d, 3 * d); self.proj = nn.Linear(d, d)
        self.ln2 = nn.LayerNorm(d); self.fc = nn.Linear(d, 4 * d); self.fc2 = nn.Linear(4 * d, d)
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(self.ln1(x)).split(C, dim=2)
        q, k, v = [t.view(B, T, self.h, C // self.h).transpose(1, 2) for t in (q, k, v)]
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        x = x + self.proj(y.transpose(1, 2).contiguous().view(B, T, C))
        return x + self.fc2(F.gelu(self.fc(self.ln2(x))))

class GPT(nn.Module):
    def __init__(self, V=CFG['vocab'], d=CFG['d_model'], L=CFG['n_layer'], h=CFG['n_head'], T=CFG['ctx']):
        super().__init__()
        self.wte = nn.Embedding(V, d); self.wpe = nn.Embedding(T, d)
        self.blocks = nn.ModuleList([Block(d, h) for _ in range(L)])
        self.lnf = nn.LayerNorm(d)
    def forward(self, idx, targets):
        x = self.wte(idx) + self.wpe(torch.arange(idx.shape[1], device=idx.device))
        for b in self.blocks:
            x = b(x)
        logits = F.linear(self.lnf(x), self.wte.weight)
        return F.cross_entropy(logits.view(-1, logits.size(-1)), targets.reshape(-1))

def make_model():
    return GPT().to(DEV)

def newton_schulz5(G, steps):
    a, b, c = 3.4445, -4.7750, 2.0315
    X = G.float()
    X = X / (X.norm() + 1e-7)
    tr = X.size(0) > X.size(1)
    if tr: X = X.T
    for _ in range(steps):
        A = X @ X.T
        B = b * A + c * (A @ A)
        X = a * X + B @ X
    return X.T if tr else X

class Muon(torch.optim.Optimizer):
    def __init__(self, params, lr=0.02, momentum=0.95, nesterov=True, ns_steps=5):
        super().__init__(params, dict(lr=lr, momentum=momentum, nesterov=nesterov, ns_steps=ns_steps))
    @torch.no_grad()
    def step(self, closure=None):
        for grp in self.param_groups:
            for p in grp['params']:
                if p.grad is None: continue
                st = self.state[p]
                if 'buf' not in st: st['buf'] = torch.zeros_like(p.grad)
                buf = st['buf']; buf.mul_(grp['momentum']).add_(p.grad)
                u = p.grad.add(buf, alpha=grp['momentum']) if grp['nesterov'] else buf
                u = newton_schulz5(u, grp['ns_steps'])
                u *= max(1.0, p.size(0) / p.size(1)) ** 0.5
                p.add_(u.to(p.dtype), alpha=-grp['lr'])

def make_muon_opts(model, muon_lr, adam_lr, wd):
    hidden = [p for p in model.blocks.parameters() if p.ndim == 2]
    hid = {id(p) for p in hidden}
    rest = [p for p in model.parameters() if id(p) not in hid]
    muon = Muon(hidden, lr=muon_lr, momentum=CFG['muon_momentum'], ns_steps=CFG['ns_steps'])
    adam = torch.optim.AdamW([{'params': [p for p in rest if p.ndim >= 2], 'weight_decay': wd},
                              {'params': [p for p in rest if p.ndim < 2], 'weight_decay': 0.0}],
                             lr=adam_lr, betas=(0.9, 0.95), eps=1e-8)
    for o, lr in ((muon, muon_lr), (adam, adam_lr)):
        for g in o.param_groups: g['base_lr'] = lr
    return [muon, adam]

def cos_fn(total, warm):
    def f(s):
        if s < warm: return (s + 1) / warm
        return 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm)))
    return f

def train_step(model, opts, scaler, x, y):
    with torch.autocast('cuda', dtype=torch.float16):
        loss = model(x, y)
    model.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    for o in opts: scaler.unscale_(o)
    torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
    for o in opts: scaler.step(o)
    scaler.update()

@torch.no_grad()
def eval_loss(model, ev):
    model.eval(); X, Y = ev; tot = 0.0; bs = CFG['eval_bs']
    for i in range(0, len(X), bs):
        with torch.autocast('cuda', dtype=torch.float16):
            tot += model(X[i:i + bs], Y[i:i + bs]).float().item() * len(X[i:i + bs])
    model.train()
    return tot / len(X)

def delta_rel(sd_a, sd_b):
    num = den = 0.0
    for k, v in sd_a.items():
        if not v.is_floating_point(): continue
        num += (sd_b[k].float() - v.float()).pow(2).sum().item(); den += v.float().pow(2).sum().item()
    return math.sqrt(num / den)

def baby_at(curve, target):
    """코드 loss가 target까지 내려간 지점의 BabyLM loss (선형 보간)"""
    for a, b in zip(curve, curve[1:]):
        if b['code'] <= target:
            t = 1.0 if a['code'] == b['code'] else min(max((a['code'] - target) / (a['code'] - b['code']), 0), 1)
            return a['baby'] + t * (b['baby'] - a['baby'])
    return float('nan')

print('도구 준비 완료')

In [ ]:
# ============ 셀 4: L5A 사전학습 모델 6개 불러오기 + 확인 ============
def find_input(name):
    hits = glob.glob(f'/kaggle/input/**/exp_l5_pretrain/{name}', recursive=True)
    assert hits, f"{name}을 못 찾았어요. Add Input에서 L5A 노트북을 추가했는지 확인하세요."
    return hits[0]

PRE_SUM = pd.read_csv(find_input('pretrain_summary.csv')).set_index('model')
PRE_SD = {}
print('확인: 불러온 모델의 BabyLM loss가 L5A 셀 7 기록과 같아야 함')
for t in MODELS:
    PRE_SD[t] = torch.load(find_input(f'{t}.pt'), map_location=DEV)
    m = make_model(); m.load_state_dict(PRE_SD[t])
    print(f'  {t:24s} BabyLM {eval_loss(m, EV_BABY):.4f} (L5A 기록 {PRE_SUM.loc[t, "babylm_loss"]:.4f}) | '
          f'코드 loss {eval_loss(m, EV_CODE):.3f}', flush=True)
    del m
torch.cuda.empty_cache()

FT_SETTINGS = [(CFG['muon_pre_lr'] / d, a) for d, a in zip(CFG['ft_muon_div'], CFG['ft_adam_lrs'])]
STRENGTH = {round(a, 12): s for (_, a), s in zip(FT_SETTINGS, ['weak', 'strong'])}
print('\n파인튜닝 설정 (Muon lr, 보조 AdamW lr):', [(f'{m:.3g}', f'{a:.3g}') for m, a in FT_SETTINGS])

In [ ]:
# ============ 셀 5: Muon 파인튜닝 (설정 2개 × seed 3개 × 모델 6개 = 36회) ============
code_stream = Stream(TR_CODE, CFG['ft_bs'])
ft_lr_fn = cos_fn(CFG['ft_steps'], CFG['ft_warmup'])

res_path, curve_path = f'{OUT}/ft_results.csv', f'{OUT}/ft_curves.json'
ROWS = pd.read_csv(res_path).to_dict('records') if os.path.exists(res_path) else []
CURVES = json.load(open(curve_path)) if os.path.exists(curve_path) else {}
done = {(round(r['adam_lr'], 12), r['seed'], r['pre']) for r in ROWS}

total_runs = len(FT_SETTINGS) * len(CFG['ft_seeds']) * len(MODELS); k = 0; t_all = time.time()
for muon_lr, adam_lr in FT_SETTINGS:
    for s in CFG['ft_seeds']:
        for pre in MODELS:
            k += 1
            if (round(adam_lr, 12), s, pre) in done: continue
            t0 = time.time()
            seed_all(s)
            model = make_model(); model.load_state_dict(PRE_SD[pre])
            opts = make_muon_opts(model, muon_lr, adam_lr, CFG['ft_wd']); scaler = torch.amp.GradScaler('cuda')
            curve = [dict(step=0, code=eval_loss(model, EV_CODE), baby=eval_loss(model, EV_BABY))]
            model.train()
            for st in range(CFG['ft_steps']):
                f = ft_lr_fn(st)
                for o in opts:
                    for g in o.param_groups: g['lr'] = g['base_lr'] * f
                x, y = code_stream.batch(st, s)          # 같은 seed면 6개 모델 모두 같은 코드 배치
                train_step(model, opts, scaler, x, y)
                if (st + 1) % CFG['ft_eval_every'] == 0:
                    curve.append(dict(step=st + 1, code=eval_loss(model, EV_CODE), baby=eval_loss(model, EV_BABY)))
            c0, cN = curve[0], curve[-1]
            row = dict(strength=STRENGTH[round(adam_lr, 12)], muon_lr=muon_lr, adam_lr=adam_lr, seed=s, pre=pre,
                       code_before=c0['code'], code_after=cN['code'],
                       baby_before=c0['baby'], baby_after=cN['baby'], forgetting=cN['baby'] - c0['baby'],
                       delta_rel=delta_rel(PRE_SD[pre], model.state_dict()))
            ROWS.append(row); CURVES[f'{adam_lr}|{s}|{pre}'] = curve
            pd.DataFrame(ROWS).to_csv(res_path, index=False); json.dump(CURVES, open(curve_path, 'w'))
            el = time.time() - t_all
            print(f'[{k:2d}/{total_runs}] {row["strength"]:6s} seed={s} {pre:24s} | 코드 {c0["code"]:.3f}→{cN["code"]:.3f} | '
                  f'BabyLM {c0["baby"]:.3f}→{cN["baby"]:.3f} (망각 +{row["forgetting"]:.4f}) | '
                  f'{time.time()-t0:.0f}s (누적 {el/60:.0f}분)', flush=True)
            del model, opts; torch.cuda.empty_cache()
print('\n파인튜닝 완료')

In [ ]:
# ============ 셀 6: 결과 — 망각 감소와 비용 ============
import matplotlib.pyplot as plt
df = pd.DataFrame(ROWS)

# 같은 코드 loss 지점 (세기별로 18개 run이 모두 도달한 지점)
TARGET = {}
df['matched_forgetting'] = np.nan
for st in ['weak', 'strong']:
    sub = df[df.strength == st]
    TARGET[st] = max(min(p['code'] for p in CURVES[f'{r.adam_lr}|{r.seed}|{r.pre}']) for r in sub.itertuples())
    for i, r in sub.iterrows():
        df.loc[i, 'matched_forgetting'] = baby_at(CURVES[f'{r.adam_lr}|{r.seed}|{r.pre}'], TARGET[st]) - r.baby_before
df.to_csv(f'{OUT}/ft_results_final.csv', index=False)

print('===== 1. 평균 =====')
display(df.groupby(['strength', 'pre'], sort=False)[['code_after', 'baby_before', 'baby_after', 'forgetting',
                                                    'matched_forgetting', 'delta_rel']].mean().round(4))

print('\n===== 2. pre_off 대비 망각 감소 (같은 seed 짝 비교, 양수 = 덜 잊음) =====')
EFF = []
for st in ['weak', 'strong']:
    print(f'\n[{st}]  같은 코드 loss 기준점 = {TARGET[st]:.3f}')
    a = df[(df.strength == st) & (df.pre == 'pre_off')].set_index('seed').sort_index()
    for pre in MODELS[1:]:
        b = df[(df.strength == st) & (df.pre == pre)].set_index('seed').sort_index()
        d_f = a.forgetting - b.forgetting
        d_m = a.matched_forgetting - b.matched_forgetting
        red_f = 1 - b.forgetting.mean() / a.forgetting.mean()
        red_m = 1 - b.matched_forgetting.mean() / a.matched_forgetting.mean()
        cost = PRE_SUM.loc[pre, '비용_배수']
        EFF.append(dict(strength=st, model=pre, cost=cost, pre_loss=PRE_SUM.loc[pre, 'babylm_loss'],
                        red_final=red_f, red_matched=red_m, pos_final=int((d_f > 0).sum()), pos_matched=int((d_m > 0).sum()),
                        code_gain=(a.code_after - b.code_after).mean()))
        print(f'  {SHORT[pre]:16s} 비용 {cost:.2f}배 | 최종 망각 {red_f*100:+5.1f}% ({int((d_f>0).sum())}/3) | '
              f'같은 지점 {red_m*100:+5.1f}% ({int((d_m>0).sum())}/3) | seed별 {[round(x, 4) for x in d_m]} | '
              f'코드 loss 차 {(a.code_after - b.code_after).mean():+.4f}')
eff = pd.DataFrame(EFF); eff.to_csv(f'{OUT}/efficiency.csv', index=False)

print('\n===== 3. 핵심: decay 스펙트럼 0.008이 full 스펙트럼 효과의 몇 %를 얻나 (같은 지점 망각 감소 기준) =====')
for st in ['weak', 'strong']:
    e = eff[eff.strength == st].set_index('model')
    d, f = e.loc['pre_decay_spec_0.008', 'red_matched'], e.loc['pre_full_spec', 'red_matched']
    share = d / f if f > 0 else float('nan')
    print(f'  [{st}] decay {d*100:+.1f}% vs full {f*100:+.1f}% → full 대비 {share*100:.0f}%  '
          f'(추가 비용: decay {e.loc["pre_decay_spec_0.008", "cost"]-1:.2f}배, full {e.loc["pre_full_spec", "cost"]-1:.2f}배)')

# ---------- 그림 1: 망각 곡선 (seed 평균) ----------
colors = dict(zip(MODELS, ['tab:gray', 'tab:orange', '#9ecae1', '#4292c6', '#08519c', 'tab:red']))
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=False)
for ax, st in zip(axes, ['weak', 'strong']):
    adam_lr = [a for (_, a) in FT_SETTINGS if STRENGTH[round(a, 12)] == st][0]
    for pre in MODELS:
        cvs = [CURVES[f'{adam_lr}|{s}|{pre}'] for s in CFG['ft_seeds']]
        code = np.mean([[p['code'] for p in c] for c in cvs], 0); baby = np.mean([[p['baby'] for p in c] for c in cvs], 0)
        ax.plot(code, baby, color=colors[pre], marker='o', ms=2.5, lw=1.4, label=SHORT[pre])
    ax.axvline(TARGET[st], color='k', ls=':', lw=0.8); ax.invert_xaxis(); ax.grid(alpha=0.3)
    ax.set_xlabel('code val loss (lower = learned more)'); ax.set_ylabel('BabyLM val loss (higher = forgot more)')
    ax.set_title(f'Muon fine-tune, strength = {st} (mean of 3 seeds)'); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f'{OUT}/forgetting_curves.png', dpi=150); plt.show()

# ---------- 그림 2: 비용 대비 망각 감소 ----------
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, st in zip(axes, ['weak', 'strong']):
    e = eff[eff.strength == st]
    ax.scatter([1.0], [0.0], color=colors['pre_off'], s=60, zorder=3); ax.annotate('off', (1.0, 0.0), textcoords='offset points', xytext=(5, 5), fontsize=8)
    for r in e.itertuples():
        ax.scatter(r.cost, r.red_matched * 100, color=colors[r.model], s=60, zorder=3)
        ax.annotate(SHORT[r.model], (r.cost, r.red_matched * 100), textcoords='offset points', xytext=(5, 5), fontsize=8)
    ax.axhline(0, color='k', lw=0.8); ax.grid(alpha=0.3)
    ax.set_xlabel('pretraining cost (x of no-SAM)'); ax.set_title(f'strength = {st}')
axes[0].set_ylabel('forgetting reduction at matched code loss (%)')
plt.tight_layout(); plt.savefig(f'{OUT}/cost_vs_reduction.png', dpi=150); plt.show()

json.dump(dict(targets=TARGET, efficiency=EFF), open(f'{OUT}/summary.json', 'w'), indent=2, default=float)
print('저장:', OUT)